# 🇰🇪 Child Mortality Recommendation System — Kenya
## Notebook 01: Data Preparation & Exploratory Data Analysis
---
**CRISP-DM Phases:** Data Understanding + Data Preparation  
**Day:** 1  
**Goal:** Clean all three datasets, engineer composite features, and explore patterns in Kenya county-level child mortality data.

## 1.0 Environment Setup

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so src/ modules are importable
sys.path.insert(0, os.path.abspath('..'))

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

TIER_PALETTE = {'High': '#D62728', 'Medium': '#FF7F0E', 'Low': '#2CA02C'}
os.makedirs('../visualizations', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('✓ Environment ready')
print(f'  Python  : {sys.version.split()[0]}')
print(f'  Pandas  : {pd.__version__}')
print(f'  NumPy   : {np.__version__}')

## 2.0 Import src Modules

In [ ]:
from src.mortality_cleaner import MortalityCleaner
from src.feature_engineer  import FeatureEngineer

print('✓ src modules imported successfully')

## 3.0 Data Loading & Cleaning

Using `MortalityCleaner` to load, validate, and clean all three raw datasets from `data/raw/`.

In [ ]:
cleaner = MortalityCleaner(data_dir='../data/raw')

counties_df      = cleaner.clean_county_indicators()
interventions_df = cleaner.clean_interventions()
deployments_df   = cleaner.clean_deployments()

cleaner.save_cleaned(output_dir='../data/processed')

### 3.1 Cleaning Summary

In [ ]:
cleaner.summary()

### 3.2 Preview — County Mortality Indicators

In [ ]:
print(f'Shape: {counties_df.shape}')
counties_df.head(10)

### 3.3 Preview — Intervention Registry

In [ ]:
print(f'Shape: {interventions_df.shape}')
interventions_df

### 3.4 Preview — Deployment Records

In [ ]:
print(f'Shape: {deployments_df.shape}')
deployments_df.head(10)

## 4.0 Feature Engineering

Building four composite scores:
- **Health System Score** — skilled attendance, immunization, ANC, facility delivery, distance
- **Nutrition Risk Score** — stunting (60%) + wasting (40%)
- **WASH Score** — clean water (60%) + sanitation (40%)
- **Deprivation Index** — poverty (70%) + education deficit (30%)

In [ ]:
fe = FeatureEngineer(scale_features=True)

# Build composite scores
counties_df = fe.build_composite_scores(counties_df)

# Encode categoricals
counties_df = fe.encode_categoricals(counties_df, columns=['Risk_Tier', 'Region'])

print('\nComposite scores added:')
composite_cols = ['Health_System_Score', 'Nutrition_Risk_Score', 'WASH_Score', 'Deprivation_Index']
counties_df[['County', 'Year', 'Risk_Tier'] + composite_cols].head(10)

### 4.1 Composite Score Summary by Risk Tier

In [ ]:
fe.score_summary(counties_df)

### 4.2 Build County Feature Matrix (for collaborative filtering)

In [ ]:
county_matrix = fe.build_county_feature_matrix(counties_df, year=2022)
county_matrix.to_csv('../data/processed/county_feature_matrix.csv')

print(f'Feature matrix shape: {county_matrix.shape}')
county_matrix.head()

## 5.0 Overview Statistics

In [ ]:
df_2022 = counties_df[counties_df['Year'] == 2022].copy()

print('=== Dataset Overview ===')
print(f'Total counties     : {df_2022["County"].nunique()}')
print(f'Years covered      : {sorted(counties_df["Year"].unique())}')
print(f'ASAL counties      : {int(df_2022["ASAL_Flag"].sum())}')
print(f'Non-ASAL counties  : {int((df_2022["ASAL_Flag"] == 0).sum())}')
print()
print('Under-5 Mortality Rate (2022) — Summary Statistics:')
display(df_2022['Under5_Mortality_Rate_per1000'].describe().round(2).to_frame())

### 5.1 Highest & Lowest Mortality Counties

In [ ]:
print('Top 5 — Highest mortality:')
display(df_2022.nlargest(5, 'Under5_Mortality_Rate_per1000')[
    ['County','Region','Risk_Tier','Under5_Mortality_Rate_per1000','ASAL_Flag']
].reset_index(drop=True))

print('\nTop 5 — Lowest mortality:')
display(df_2022.nsmallest(5, 'Under5_Mortality_Rate_per1000')[
    ['County','Region','Risk_Tier','Under5_Mortality_Rate_per1000','ASAL_Flag']
].reset_index(drop=True))

## 6.0 EDA Visualisations

### 6.1 Mortality Distribution by Risk Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Under-5 Mortality Rate Distribution — Kenya Counties (2022)',
             fontsize=14, fontweight='bold')

for tier, color in TIER_PALETTE.items():
    subset = df_2022[df_2022['Risk_Tier'] == tier]['Under5_Mortality_Rate_per1000']
    axes[0].hist(subset, bins=8, color=color, alpha=0.75, label=tier, edgecolor='white')
axes[0].set_xlabel('Mortality Rate per 1,000 Live Births')
axes[0].set_ylabel('Number of Counties')
axes[0].set_title('Distribution by Risk Tier')
axes[0].legend(title='Risk Tier')

counts = df_2022['Risk_Tier'].value_counts().reindex(['High','Medium','Low'])
bars = axes[1].bar(counts.index, counts.values,
                   color=[TIER_PALETTE[t] for t in counts.index], edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Risk Tier')
axes[1].set_ylabel('Number of Counties')
axes[1].set_title('County Count per Risk Tier')

plt.tight_layout()
plt.savefig('../visualizations/01_mortality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Regional Mortality Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
region_stats = (
    df_2022.groupby('Region')['Under5_Mortality_Rate_per1000']
    .agg(['mean','min','max']).sort_values('mean', ascending=False)
)
bars = ax.bar(region_stats.index, region_stats['mean'],
              color='#4C72B0', alpha=0.85, edgecolor='white')
ax.errorbar(region_stats.index, region_stats['mean'],
            yerr=[region_stats['mean']-region_stats['min'],
                  region_stats['max']-region_stats['mean']],
            fmt='none', color='#333', capsize=5, linewidth=1.2)
for bar, val in zip(bars, region_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}', ha='center', fontsize=9.5)
ax.set_title('Average Under-5 Mortality Rate by Region (2022)\n(Error bars = min–max range)')
ax.set_ylabel('Rate per 1,000 Live Births')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../visualizations/02_mortality_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 ASAL vs Non-ASAL Counties

In [ ]:
df_2022['County Type'] = df_2022['ASAL_Flag'].map({1:'ASAL', 0:'Non-ASAL'})
palette = {'ASAL':'#D62728', 'Non-ASAL':'#2CA02C'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ASAL vs Non-ASAL Counties (2022)', fontsize=14, fontweight='bold')

sns.boxplot(data=df_2022, x='County Type', y='Under5_Mortality_Rate_per1000',
            palette=palette, ax=axes[0], width=0.45)
sns.stripplot(data=df_2022, x='County Type', y='Under5_Mortality_Rate_per1000',
              palette=palette, ax=axes[0], alpha=0.45, jitter=True, size=5)
axes[0].set_title('Under-5 Mortality Rate')
axes[0].set_ylabel('Rate per 1,000 Live Births')
axes[0].set_xlabel('')

indicators = {
    'Skilled_Birth_Attendance_pct': 'Skilled Birth Attendance',
    'Immunization_Coverage_pct':    'Immunization Coverage',
    'Clean_Water_Access_pct':       'Clean Water Access',
}
for col, label in indicators.items():
    if col in df_2022.columns:
        means = df_2022.groupby('County Type')[col].mean()
        axes[1].plot(means.index, means.values, marker='o', linewidth=2.5,
                     markersize=9, label=label)
axes[1].set_title('Health Indicator Comparison')
axes[1].set_ylabel('Coverage (%)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../visualizations/03_asal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Correlation Heatmap

In [ ]:
corr_cols = [
    'Under5_Mortality_Rate_per1000', 'Poverty_Index',
    'Skilled_Birth_Attendance_pct',  'Clean_Water_Access_pct',
    'Immunization_Coverage_pct',     'Stunting_Prevalence_pct',
    'Female_Literacy_Rate_pct',      'Health_System_Score',
    'WASH_Score',                    'Deprivation_Index',
]
available = [c for c in corr_cols if c in df_2022.columns]
corr_matrix = df_2022[available].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.4,
            ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Indicators vs Mortality Rate', fontsize=13, pad=14)
plt.tight_layout()
plt.savefig('../visualizations/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop correlates with Under-5 Mortality:')
display(corr_matrix['Under5_Mortality_Rate_per1000']
        .drop('Under5_Mortality_Rate_per1000').abs()
        .sort_values(ascending=False).head(5).round(3).to_frame('|Correlation|'))

### 6.5 Year-on-Year Mortality Trend

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for tier, color in TIER_PALETTE.items():
    trend = (counties_df[counties_df['Risk_Tier'] == tier]
             .groupby('Year')['Under5_Mortality_Rate_per1000'].mean())
    ax.plot(trend.index, trend.values, marker='o', color=color,
            linewidth=2.5, markersize=9, label=f'{tier} Risk')
    for x, y in zip(trend.index, trend.values):
        ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=9.5, color=color)
ax.set_title('Under-5 Mortality Trend by Risk Tier (2022–2024)')
ax.set_ylabel('Rate per 1,000 Live Births')
ax.set_xticks([2022, 2023, 2024])
ax.legend(title='Risk Tier')
plt.tight_layout()
plt.savefig('../visualizations/05_mortality_trend.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.6 Intervention Composite Scores

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
df_int = interventions_df.sort_values('Composite_Score', ascending=True)
categories = df_int['Category'].unique()
cmap = plt.cm.get_cmap('tab10', len(categories))
cat_color = dict(zip(categories, [cmap(i) for i in range(len(categories))]))
bar_colors = [cat_color[c] for c in df_int['Category']]

bars = ax.barh(df_int['Intervention_Name'], df_int['Composite_Score'],
               color=bar_colors, edgecolor='white', height=0.7)
for bar, val in zip(bars, df_int['Composite_Score']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
handles = [plt.Rectangle((0,0),1,1, color=c) for c in cat_color.values()]
ax.legend(handles, cat_color.keys(), title='Category', loc='lower right', fontsize=9)
ax.set_title('Intervention Composite Scores\n(Effectiveness 45% + Feasibility 30% + Cost-Effectiveness 25%)')
ax.set_xlabel('Composite Score (0–100)')
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig('../visualizations/06_intervention_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.0 Key Findings Summary

In [ ]:
asal_mean    = df_2022[df_2022['ASAL_Flag']==1]['Under5_Mortality_Rate_per1000'].mean()
nonasal_mean = df_2022[df_2022['ASAL_Flag']==0]['Under5_Mortality_Rate_per1000'].mean()

print('=' * 55)
print('  KEY EDA FINDINGS')
print('=' * 55)
print(f'  ASAL mean mortality      : {asal_mean:.1f} per 1,000')
print(f'  Non-ASAL mean mortality  : {nonasal_mean:.1f} per 1,000')
print(f'  ASAL / Non-ASAL ratio    : {asal_mean/nonasal_mean:.1f}×')
print()
print(f'  Strongest intervention   : {interventions_df.nlargest(1,"Composite_Score")["Intervention_Name"].values[0]}')
print(f'  Composite score          : {interventions_df["Composite_Score"].max():.1f}')
print()
print('  Next step → notebooks/02_modeling.ipynb')
print('=' * 55)